# 05b · Charges des boutiques : intégration du plan stratégique

**Projet 4 : Analyse des données de vente et retours** · US35 et US36

**Objectif** : intégrer les charges des boutiques (plan stratégique de Contoso) en suivant la même démarche que pour les ventes : chargement, mise en forme, diagnostic, variables, puis une table commune « ventes et charges » par boutique et par mois.

| Étape | Ce qu'on fait | Question posée |
|---|---|---|
| 1. Chargement | Lire le plan extrait et ses trois tables de description | Que contient le plan ? |
| 2. Mise en forme (ETL) | Types, dates, libellés en français, familles de comptes | Les colonnes sont-elles exploitables ? |
| 3. Diagnostic | Complétude, doublons, valeurs impossibles, rapprochement avec nos ventes | Peut-on faire confiance au plan, et correspond-il à nos ventes ? |
| 4. Nature des données | Lien entre réel, budget et prévision ; lien entre charges et ventes, année par année | Le budget est-il un vrai budget ? Les charges suivent-elles les ventes ? |
| 5. Nettoyage | Décisions | Y a-t-il quelque chose à corriger ? |
| 6. Variables | Compte de résultat par boutique, mois et scénario | Que reste-t-il une fois les charges payées ? |
| 7. Table commune | Ventes et charges réunies par boutique et par mois | Quelle est la rentabilité nette de chaque boutique ? |
| 8. Validation | Cohérence des totaux | |
| 9. Enregistrement | | |

**Lit dans** : `data/raw/contoso/Plan/FactStrategyPlan.csv` (produit par `src/extraction.py`), `data/raw/contoso/Dim` (`DimAccount`, `DimScenario`, `DimProductCategory`), `data/clean` (ventes avec variables, dimension boutique)

**Écrit dans** : `data/clean/fait_charges.parquet`, `data/clean/dimensions` (`dim_compte`, `dim_scenario`, `dim_categorie`), `data/clean/agregats` (`resultat_boutique_mois`, `ventes_charges_boutique_mois`), `reports/figures`

**Périmètre** : nos 56 boutiques sur trois années. 2019 est l'année de référence, 2021 l'exercice étudié (`config.ANNEE_REFERENCE`, `config.EXERCICE`). Les tableaux de résultat sont donnés **année par année** : un cumul sur trois ans mélangerait une année où les magasins gagnent de l'argent et une année où ils en perdent.

**Ce notebook ne modifie rien de ce qui existe.** Les notebooks `00` à `05` et la table des ventes restent tels quels.

### Pourquoi une deuxième table de faits

Les charges ne peuvent pas être ajoutées comme des colonnes de la table des ventes, parce qu'elles ne sont pas au même niveau de détail :

| | Ventes (`fait_ventes`) | Charges (`fait_charges`) |
|---|---|---|
| Une ligne = | un produit, une boutique, un jour, une promotion | une boutique, un mois, un compte, un scénario, une catégorie de produit |
| Dimensions communes | boutique, date, catégorie de produit | boutique, mois, catégorie de produit |
| Dimensions propres | produit, promotion, canal | compte, scénario |

Un loyer mensuel n'appartient à aucune vente en particulier. Le modèle passe donc d'un flocon (une table de faits) à une **constellation** : deux tables de faits reliées par leurs dimensions communes. Pour les comparer, on ramène les ventes au niveau boutique et mois (section 7).

## 0. Préparation

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# 1. Retrouver la racine du projet (le dossier qui contient src/ et data/)
# ---------------------------------------------------------------------------
racine = Path.cwd()
while not ((racine / "src").is_dir() and (racine / "data").is_dir()):
    if racine == racine.parent:
        raise FileNotFoundError("Racine du projet introuvable : lancer ce notebook depuis le dossier Projet ADD ou un de ses sous-dossiers.")
    racine = racine.parent
sys.path.insert(0, str(racine))

# ---------------------------------------------------------------------------
# 2. Modules du projet
# ---------------------------------------------------------------------------
#   config -> chemins (CONTOSO, DIM, CLEAN, AGREGATS)
#   viz    -> charte graphique
from src import config, viz

config.creer_dossiers()
viz.appliquer_style()
pd.set_option("display.max_columns", 70)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Racine du projet :", config.ROOT)

# Dossier du plan stratégique, créé par src/extraction.py
DOSSIER_PLAN = config.PLAN

# Rôle des années, lu dans config
EXERCICE, REFERENCE = config.EXERCICE, config.ANNEE_REFERENCE

# Tolérance pour comparer des montants (arrondis au cent dans la source)
TOLERANCE = 0.02

# ---------------------------------------------------------------------------
# 3. Registre des contrôles (même principe que dans 03_diagnostic)
# ---------------------------------------------------------------------------
registre = []


def noter(controle, nb, gravite, action, total=None):
    """Ajoute un contrôle au registre et l'affiche.

    controle : ce qui est vérifié
    nb       : nombre de cas concernés (0 = rien à signaler)
    gravite  : "OK" si nb vaut 0, sinon la gravité donnée ("Information", "À décider", "À corriger")
    action   : ce qu'on propose de faire
    total    : nombre total de cas, pour afficher une part
    """
    nb = int(nb)
    gravite = "OK" if nb == 0 else gravite
    part = f" ({nb / total:.2%})" if total else ""
    registre.append({"controle": controle, "nb": nb, "gravite": gravite, "action": action})
    nombre = f"{nb:,}".replace(",", " ")   # séparateur de milliers à la française, sur le nombre seulement
    print(f"[{gravite:<11}] {controle} : {nombre}{part}")

## 1. Chargement

Le plan stratégique a été extrait de Contoso par `src/extraction.py`, pour nos 56 boutiques et nos trois années. Trois petites tables le décrivent :

- `DimAccount` : les **comptes** (ventes, coût des ventes, personnel, loyers, publicité...) ;
- `DimScenario` : les **scénarios** (réel, budget, prévision) ;
- `DimProductCategory` : les catégories de produits, déjà utilisées pour les ventes.

In [ ]:
chemin_plan = DOSSIER_PLAN / config.FICHIER_PLAN
if not chemin_plan.exists():
    raise FileNotFoundError(f"{chemin_plan} introuvable : lancer d'abord python -m src.extraction depuis le dossier du projet.")

# utf-8-sig : le fichier a été écrit avec un marqueur de début (BOM), comme nos autres CSV
plan_brut = pd.read_csv(chemin_plan, encoding="utf-8-sig")
comptes_brut = pd.read_csv(config.DIM / "DimAccount.csv", encoding="utf-8-sig")
scenarios_brut = pd.read_csv(config.DIM / "DimScenario.csv", encoding="utf-8-sig")
categories_brut = pd.read_csv(config.DIM / "DimProductCategory.csv", encoding="utf-8-sig")

print(f"Plan stratégique : {len(plan_brut):,} lignes × {plan_brut.shape[1]} colonnes".replace(",", " "))
print("Colonnes :", list(plan_brut.columns))
display(plan_brut.head(3))

print("\nNombre de valeurs différentes par colonne :")
display(plan_brut.nunique().to_frame("valeurs distinctes").T)

print("Scénarios :")
display(scenarios_brut[["ScenarioKey", "ScenarioName"]])
print("Comptes (la colonne ParentAccountKey donne le compte parent : c'est une hiérarchie) :")
display(comptes_brut[["AccountKey", "ParentAccountKey", "AccountName", "AccountType"]])

## 2. Mise en forme (ETL)

Mêmes principes que dans `01_etl` et `02_flocon` : des noms de colonnes en français, des dates de type date, des libellés lisibles.

**Les comptes.** Le plan n'utilise que les comptes de détail (17 sur 24). Les autres sont des totaux de la hiérarchie (par exemple « Expense », « Marketing Cost »), que le plan ne renseigne pas. On range les 17 comptes en **cinq familles**, qui sont celles d'un compte de résultat :

| Famille | Comptes |
|---|---|
| Ventes | Sale Revenue |
| Coût des ventes | Cost of Goods Sold |
| Frais généraux | administration, informatique, personnel, énergie et communication, loyers, autres |
| Publicité | campagnes de printemps, de la rentrée, entreprises, été ; campagne des Fêtes par média (radio et télévision, presse, internet, autres) |
| Impôts | Taxation |

**Les dates.** Le plan est mensuel : chaque date est le premier jour d'un mois. La colonne `Date` est déjà décalée de 12 ans par le script d'extraction, comme pour les ventes.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Dimension des comptes : libellé français et famille
# ---------------------------------------------------------------------------
# Clé du compte -> (libellé français, famille). Seuls les comptes utilisés par le plan sont listés.
COMPTES = {
    4:  ("Ventes", "Ventes"),
    5:  ("Coût des ventes", "Coût des ventes"),
    7:  ("Administration", "Frais généraux"),
    8:  ("Informatique", "Frais généraux"),
    9:  ("Personnel", "Frais généraux"),
    10: ("Énergie et communication", "Frais généraux"),
    11: ("Loyers et immobilier", "Frais généraux"),
    12: ("Autres charges", "Frais généraux"),
    15: ("Publicité de printemps", "Publicité"),
    16: ("Publicité de la rentrée", "Publicité"),
    17: ("Publicité entreprises", "Publicité"),
    18: ("Publicité d'été", "Publicité"),
    20: ("Publicité des Fêtes : radio et télévision", "Publicité"),
    21: ("Publicité des Fêtes : presse", "Publicité"),
    22: ("Publicité des Fêtes : internet", "Publicité"),
    23: ("Publicité des Fêtes : autres médias", "Publicité"),
    19: ("Impôts et taxes", "Impôts"),
}
# Ordre d'affichage des familles : celui d'un compte de résultat
FAMILLES = ["Ventes", "Coût des ventes", "Frais généraux", "Publicité", "Impôts"]

dim_compte = comptes_brut[["AccountKey", "AccountName"]].rename(
    columns={"AccountKey": "id_compte", "AccountName": "nom_compte_source"})
dim_compte = dim_compte[dim_compte.id_compte.isin(COMPTES)].copy()
dim_compte["nom_compte"] = dim_compte.id_compte.map(lambda k: COMPTES[k][0])
dim_compte["famille_compte"] = dim_compte.id_compte.map(lambda k: COMPTES[k][1])
dim_compte["nom_compte_source"] = dim_compte.nom_compte_source.str.strip()
dim_compte = dim_compte.sort_values("id_compte").reset_index(drop=True)

# Tous les comptes présents dans le plan doivent être connus (sinon il faut compléter COMPTES)
comptes_inconnus = sorted(set(plan_brut.AccountKey) - set(COMPTES))
assert not comptes_inconnus, f"Comptes du plan absents de la table COMPTES : {comptes_inconnus}"
display(dim_compte)

# ---------------------------------------------------------------------------
# 2. Dimension des scénarios : libellé français
# ---------------------------------------------------------------------------
SCENARIOS = {"Actual": "Réel", "Budget": "Budget", "Forecast": "Prévision"}
dim_scenario = scenarios_brut[["ScenarioKey", "ScenarioName"]].rename(
    columns={"ScenarioKey": "id_scenario", "ScenarioName": "nom_scenario_source"})
dim_scenario["scenario"] = dim_scenario.nom_scenario_source.map(SCENARIOS)
display(dim_scenario)

# ---------------------------------------------------------------------------
# 3. Catégories de produits : mêmes libellés que dans la table des ventes (espaces retirés)
# ---------------------------------------------------------------------------
categories = categories_brut[["ProductCategoryKey", "ProductCategoryName"]].rename(
    columns={"ProductCategoryKey": "id_categorie", "ProductCategoryName": "categorie"})
categories["categorie"] = categories.categorie.str.strip()

# Cette table devient une dimension à part entière (dim_categorie). Dans la table des ventes, la
# catégorie est une colonne de dim_produit (une ligne par produit). Les charges, elles, sont au
# niveau de la catégorie : il leur faut une table avec UNE ligne par catégorie pour s'y relier.
# On vérifie que ses libellés sont exactement ceux de dim_produit, pour que les deux se rejoignent.
dim_categorie = categories.sort_values("id_categorie").reset_index(drop=True)
dim_produit = pd.read_parquet(config.CLEAN_DIMENSIONS / "dim_produit.parquet")
categories_produits = set(dim_produit.categorie.dropna().str.strip())
assert categories_produits <= set(dim_categorie.categorie), (
    f"Catégories de dim_produit absentes de dim_categorie : {sorted(categories_produits - set(dim_categorie.categorie))}")
display(dim_categorie)

# ---------------------------------------------------------------------------
# 4. Table de faits des charges : colonnes renommées, typées, libellés joints
# ---------------------------------------------------------------------------
charges = plan_brut.rename(columns={
    "IdBoutique": "id_boutique", "Date": "date", "ScenarioKey": "id_scenario",
    "AccountKey": "id_compte", "ProductCategoryKey": "id_categorie", "Montant": "montant"})
charges["date"] = pd.to_datetime(charges["date"], format="%Y-%m-%d")       # premier jour du mois
charges["annee"] = charges.date.dt.year
charges["mois"] = charges.date.dt.month
charges["annee_mois"] = charges.date.dt.strftime("%Y-%m")                   # même format que dans 05_variables

# Jointures avec les trois dimensions. validate="many_to_one" : erreur si une clé est en double.
charges = (charges
           .merge(dim_compte[["id_compte", "nom_compte", "famille_compte"]], on="id_compte", how="left", validate="many_to_one")
           .merge(dim_scenario[["id_scenario", "scenario"]], on="id_scenario", how="left", validate="many_to_one")
           .merge(categories, on="id_categorie", how="left", validate="many_to_one"))

# Nom et pays de la boutique, depuis la dimension propre des ventes
dim_boutique = pd.read_parquet(config.CLEAN_DIMENSIONS / "dim_boutique.parquet")
colonnes_boutique = [c for c in ["id_boutique", "nom_boutique", "type_boutique", "pays"] if c in dim_boutique.columns]
charges = charges.merge(dim_boutique[colonnes_boutique], on="id_boutique", how="left", validate="many_to_one")

# Colonnes dans un ordre lisible ; les clés techniques de Contoso (EntityKey, CurrencyKey, DateContoso) sont laissées de côté
charges = charges[["date", "annee", "mois", "annee_mois", "id_boutique", "nom_boutique", "type_boutique", "pays",
                   "id_scenario", "scenario", "id_compte", "nom_compte", "famille_compte",
                   "id_categorie", "categorie", "montant"]]
print(f"Table des charges : {len(charges):,} lignes × {charges.shape[1]} colonnes".replace(",", " "))
display(charges.head(3))

## 3. Diagnostic

Mêmes familles de contrôles que pour les ventes (`03_diagnostic`), adaptées à cette table :

1. **structure** : valeurs manquantes, jointures réussies, doublons, montants impossibles ;
2. **complétude** : chaque boutique a-t-elle tous ses mois, ses scénarios et ses comptes ?
3. **rapprochement** : les ventes et le coût des ventes du plan sont-ils ceux de notre table des ventes ?

In [ ]:
# ---------------------------------------------------------------------------
# 1. Structure
# ---------------------------------------------------------------------------
noter("Cellules vides dans la table des charges", charges.isna().sum().sum(), "À corriger",
      "Vérifier les jointures avec les dimensions", total=charges.size)
CLE = ["id_boutique", "date", "id_scenario", "id_compte", "id_categorie"]
noter("Lignes en double sur la clé (boutique, mois, scénario, compte, catégorie)", charges.duplicated(CLE).sum(),
      "À corriger", "Dédoublonner", total=len(charges))
noter("Montants négatifs", (charges.montant < 0).sum(), "À décider", "Vérifier le compte", total=len(charges))
noter("Montants nuls", (charges.montant == 0).sum(), "Information", "Aucune", total=len(charges))
noter("Devises différentes dans le plan (0 = une seule devise)", plan_brut.CurrencyKey.nunique() - 1, "À décider",
      "Convertir les montants")
noter("Dates qui ne sont pas un premier jour du mois", (charges.date.dt.day != 1).sum(), "À corriger", "Ramener au mois")

# ---------------------------------------------------------------------------
# 2. Complétude : mois couverts par boutique, comparés aux mois où la boutique vend
# ---------------------------------------------------------------------------
ventes = pd.read_parquet(config.CLEAN / "fait_ventes_variables.parquet")
ventes["date_mois"] = ventes.date.dt.to_period("M").dt.to_timestamp()      # premier jour du mois de la vente

mois_plan = charges.groupby("id_boutique").date.agg(premier_mois_plan="min", nb_mois_plan="nunique")
mois_ventes = ventes.groupby("id_boutique").date_mois.agg(premier_mois_ventes="min", nb_mois_ventes="nunique")
couverture = mois_plan.join(mois_ventes, how="outer")
print("\nBoutiques des ventes :", ventes.id_boutique.nunique(), "· boutiques du plan :", charges.id_boutique.nunique())
noter("Boutiques dont les mois du plan diffèrent des mois de vente",
      (couverture.nb_mois_plan != couverture.nb_mois_ventes).sum(), "À décider", "Vérifier l'extraction", total=len(couverture))
nb_mois_periode = charges.date.nunique()
print(f"Boutiques sans les {nb_mois_periode} mois (mêmes mois que dans les ventes) :")
display(couverture[couverture.nb_mois_plan < nb_mois_periode].join(dim_boutique.set_index("id_boutique")[["nom_boutique", "date_ouverture"]]))

# Les six magasins ouverts pendant la période portent quelques ventes datées d'avant leur ouverture
# (règle N8 de 04_nettoyage). Le plan leur attribue-t-il des charges ces mois-là ?
debut_periode = charges.date.min()
ouverture_mois = dim_boutique.set_index("id_boutique").date_ouverture.dt.to_period("M").dt.to_timestamp()
reel_tmp = charges[charges.scenario == "Réel"]
avant = reel_tmp[reel_tmp.date < reel_tmp.id_boutique.map(ouverture_mois)]
charges_avant_ouverture = (avant.pivot_table(index="nom_boutique", columns="famille_compte", values="montant", aggfunc="sum", fill_value=0)
                           .join(avant.groupby("nom_boutique").date.nunique().rename("mois avant ouverture")))
print("Charges du plan sur les mois d'avant l'ouverture (scénario réel) :")
display(charges_avant_ouverture[["mois avant ouverture", "Ventes", "Frais généraux", "Publicité"]]
        .style.format({"Ventes": "{:,.0f}", "Frais généraux": "{:,.0f}", "Publicité": "{:,.0f}"}))
noter("Mois-boutiques avec des charges avant l'ouverture du magasin", avant.groupby(["id_boutique", "date"]).ngroups,
      "Information", "Garder (totaux du plan) ; comparer les magasins sur leurs années d'ouverture complète",
      total=reel_tmp.groupby(["id_boutique", "date"]).ngroups)

# Les trois scénarios sont-ils présents pour chaque boutique, mois, compte et catégorie ?
par_cellule = charges.groupby(["id_boutique", "date", "id_compte", "id_categorie"]).id_scenario.nunique()
noter("Cellules (boutique, mois, compte, catégorie) sans les trois scénarios", (par_cellule != 3).sum(), "À décider",
      "Comparer réel et budget seulement sur les cellules complètes", total=len(par_cellule))

# Quels comptes sont renseignés chaque mois ? (scénario réel)
reel = charges[charges.scenario == "Réel"]
nb_mois_boutiques = reel.groupby(["id_boutique", "date"]).ngroups
presence = (reel.groupby(["famille_compte", "nom_compte"])
            .apply(lambda g: g.groupby(["id_boutique", "date"]).ngroups / nb_mois_boutiques, include_groups=False)
            .rename("part des mois-boutiques renseignés").reset_index())
presence["famille_compte"] = pd.Categorical(presence.famille_compte, FAMILLES, ordered=True)
print("Part des mois-boutiques où chaque compte est renseigné (scénario réel) :")
display(presence.sort_values(["famille_compte", "nom_compte"]).style.format({"part des mois-boutiques renseignés": "{:.0%}"}))
# Lecture : les ventes, le coût des ventes, les charges d'exploitation et les impôts sont présents
# tous les mois. Les comptes de publicité ne le sont que pendant leur campagne : c'est normal.

### Rapprochement avec nos ventes

C'est le contrôle le plus important : il dit si les charges du plan et nos ventes décrivent bien la **même activité**. On compare, boutique par boutique et mois par mois, les ventes et le coût des ventes du plan (scénario réel) aux totaux de notre table des ventes.

Pour le coût, notre table en contient trois : `cout_total` (la colonne d'origine de la source), `cout_brut` et `cout_net` (recalculés dans `05_variables`). On regarde lequel correspond au plan.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Nos ventes, ramenées au niveau boutique et mois
# ---------------------------------------------------------------------------
ventes_mois = (ventes.groupby(["id_boutique", "date_mois"])
               .agg(ca_brut=("ca_brut", "sum"), cout_total=("cout_total", "sum"),
                    cout_brut=("cout_brut", "sum"), cout_net=("cout_net", "sum"))
               .reset_index().rename(columns={"date_mois": "date"}))

# ---------------------------------------------------------------------------
# 2. Le plan (scénario réel), au même niveau : une colonne par famille de comptes
# ---------------------------------------------------------------------------
plan_mois = (reel.pivot_table(index=["id_boutique", "date"], columns="famille_compte", values="montant", aggfunc="sum")
             .reset_index())

# how="outer" + indicator : on voit aussi les mois présents d'un seul côté
rapprochement = ventes_mois.merge(plan_mois, on=["id_boutique", "date"], how="outer", indicator=True)
print("Mois-boutiques présents des deux côtés :", int((rapprochement["_merge"] == "both").sum()),
      "· seulement dans les ventes :", int((rapprochement["_merge"] == "left_only").sum()),
      "· seulement dans le plan :", int((rapprochement["_merge"] == "right_only").sum()))

ecart_ventes = (rapprochement.ca_brut - rapprochement["Ventes"]).abs()
noter("Mois-boutiques où les ventes du plan diffèrent de notre chiffre d'affaires", (ecart_ventes > TOLERANCE).sum(),
      "À décider", "Chercher l'origine de l'écart", total=len(rapprochement))
print(f"   Plus grand écart sur un mois-boutique : {ecart_ventes.max():.4f} $")

# Garde-fou : si les ventes du plan ne sont pas notre chiffre d'affaires, les deux tables ne
# portent pas sur le même périmètre et tout ce qui suit serait faux (des charges complètes
# comparées à des ventes partielles donnent des boutiques toutes déficitaires). On s'arrête ici.
mois_ecart = int((ecart_ventes > TOLERANCE).sum() + (rapprochement["_merge"] != "both").sum())
if mois_ecart:
    raise ValueError(
        f"Les ventes du plan ne correspondent pas à notre chiffre d'affaires sur {mois_ecart} mois-boutiques "
        f"(notre chiffre d'affaires : {rapprochement.ca_brut.sum():,.0f} $ ; ventes du plan : {rapprochement['Ventes'].sum():,.0f} $). "
        "La table des ventes de data/clean ne couvre pas les mêmes années que le plan. "
        "Relancer python -m src.extraction, puis les notebooks 00 à 05 dans l'ordre avant celui-ci "
        "(05_variables doit afficher 715 291 lignes).")

# ---------------------------------------------------------------------------
# 3. Quel coût de notre table correspond au coût des ventes du plan ?
# ---------------------------------------------------------------------------
comparaison_couts = pd.DataFrame({
    "notre colonne": ["cout_total (source)", "cout_brut (05_variables)", "cout_net (05_variables)"],
    "total de la colonne": [rapprochement[c].sum() for c in ["cout_total", "cout_brut", "cout_net"]],
    "coût des ventes du plan": rapprochement["Coût des ventes"].sum(),
})
comparaison_couts["écart total"] = comparaison_couts["total de la colonne"] - comparaison_couts["coût des ventes du plan"]
comparaison_couts["mois-boutiques en écart"] = [
    int(((rapprochement[c] - rapprochement["Coût des ventes"]).abs() > TOLERANCE).sum())
    for c in ["cout_total", "cout_brut", "cout_net"]]
display(comparaison_couts.style.format({"total de la colonne": "{:,.2f}", "coût des ventes du plan": "{:,.2f}", "écart total": "{:,.2f}"}))
noter("Mois-boutiques où le coût des ventes du plan diffère de cout_total",
      ((rapprochement.cout_total - rapprochement["Coût des ventes"]).abs() > TOLERANCE).sum(),
      "À décider", "Chercher l'origine de l'écart", total=len(rapprochement))

# ---------------------------------------------------------------------------
# 4. Même contrôle au niveau le plus fin commun aux deux tables : boutique, mois, catégorie
# ---------------------------------------------------------------------------
ventes_cat = (ventes.groupby(["id_boutique", "date_mois", "categorie"]).ca_brut.sum().reset_index()
              .rename(columns={"date_mois": "date"}))
plan_cat = (reel[reel.famille_compte == "Ventes"].groupby(["id_boutique", "date", "categorie"]).montant.sum()
            .reset_index().rename(columns={"montant": "ventes_plan"}))
rapprochement_cat = ventes_cat.merge(plan_cat, on=["id_boutique", "date", "categorie"], how="outer", indicator=True)
noter("Cellules (boutique, mois, catégorie) où les ventes du plan diffèrent de notre chiffre d'affaires",
      ((rapprochement_cat.ca_brut - rapprochement_cat.ventes_plan).abs() > TOLERANCE).sum()
      + (rapprochement_cat["_merge"] != "both").sum(),
      "À décider", "Chercher l'origine de l'écart", total=len(rapprochement_cat))

# Catégories qui ont des charges un mois donné sans avoir de vente ce mois-là
cellules_charges = reel.groupby(["id_boutique", "date", "categorie"]).ngroups
print(f"   Cellules (boutique, mois, catégorie) avec des charges : {cellules_charges} · avec des ventes : {len(plan_cat)}",
      f"· charges sans vente : {cellules_charges - len(plan_cat)}")

**Ce que montre le rapprochement**

- **Les ventes du plan sont notre chiffre d'affaires**, au cent près, pour chaque boutique, chaque mois et chaque catégorie de produit. Les deux tables décrivent bien la même activité : les charges du plan peuvent être rapprochées de nos ventes sans réserve. C'est ce qui justifiait le passage aux ventes complètes (`00_inventaire`, section 9), et c'est pour garder cette égalité que la règle N6 du nettoyage ne modifie aucun montant.
- **Le coût des ventes du plan est la colonne `cout_total` de la source**, au cent près. Ce n'est ni notre `cout_brut` ni notre `cout_net`. Or le diagnostic des ventes a montré que `cout_total` est calculé de trois façons différentes sur les lignes avec retour. Le plan hérite donc de ce mélange.
- **Conséquence pour nos calculs** : la marge qu'on obtiendrait avec le coût des ventes du plan n'est pas homogène. On garde donc **nos marges** (`marge_brute`, `marge_nette`, calculées avec une seule formule), et on ne prend dans le plan que ce que nous n'avons pas : les **frais généraux**, la **publicité** et les **impôts**. L'écart entre les deux coûts est chiffré dans le tableau ci-dessus.
- Quelques catégories ont des charges un mois donné sans aucune vente ce mois-là : la boutique supporte des charges pour une catégorie qu'elle n'a pas vendue. Ce n'est pas une anomalie de jointure.

## 4. Nature des données : le budget est-il un vrai budget ?

Le diagnostic des ventes a montré que le jeu de données est généré par un programme. On pose la même question au plan : le budget et la prévision ressemblent-ils à ce qu'une entreprise aurait fixé **avant** de connaître ses ventes, ou sont-ils fabriqués à partir du réel ?

On calcule, pour chaque ligne, le rapport budget ÷ réel et prévision ÷ réel. Dans une vraie entreprise, ces rapports varient beaucoup d'une boutique et d'un mois à l'autre. On regarde aussi si les charges suivent les ventes.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Budget et prévision rapportés au réel, ligne par ligne
# ---------------------------------------------------------------------------
par_scenario = charges.pivot_table(index=["id_boutique", "date", "id_compte", "id_categorie"],
                                   columns="scenario", values="montant", aggfunc="sum")
rapports = pd.DataFrame({"budget ÷ réel": par_scenario["Budget"] / par_scenario["Réel"],
                         "prévision ÷ réel": par_scenario["Prévision"] / par_scenario["Réel"]})
display(rapports.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T.style.format("{:.4f}"))

# Les rapports les plus fréquents (arrondis à 2 décimales) et la part des lignes qu'ils couvrent
for col in rapports.columns:
    frequents = rapports[col].round(2).value_counts(normalize=True).head(5)
    print(f"{col} : valeurs les plus fréquentes ->", {float(k): f"{v:.1%}" for k, v in frequents.items()})

# ---------------------------------------------------------------------------
# 2. Les charges suivent-elles les ventes ? Corrélation par boutique et par mois (scénario réel)
# ---------------------------------------------------------------------------
suivi = plan_mois.copy()
correlations = suivi[["Coût des ventes", "Frais généraux", "Publicité", "Impôts"]].corrwith(suivi["Ventes"])
print("\nCorrélation avec les ventes du mois, par boutique et par mois :")
display(correlations.to_frame("corrélation avec les ventes").style.format("{:+.3f}"))

# Poids de chaque famille dans les ventes, sur les trois années
poids = suivi[FAMILLES].sum() / suivi["Ventes"].sum()
display(poids.to_frame("part des ventes").style.format("{:.1%}"))

# Le rapport budget ÷ réel des ventes, par année
ventes_scenarios = (charges[charges.famille_compte == "Ventes"]
                    .pivot_table(index=["id_boutique", "date"], columns="scenario", values="montant", aggfunc="sum"))
rapport_annuel = (ventes_scenarios["Budget"] / ventes_scenarios["Réel"]).groupby(ventes_scenarios.index.get_level_values("date").year).describe()
print("Ventes du budget ÷ ventes réelles, par boutique et par mois, selon l'année :")
display(rapport_annuel[["count", "mean", "std", "min", "max"]].style.format({"count": "{:.0f}", "mean": "{:.3f}", "std": "{:.3f}", "min": "{:.3f}", "max": "{:.3f}"}))

# Publicité par boutique sur les trois années : dépend-elle de la taille de la boutique ?
publicite_boutique = (reel[reel.famille_compte == "Publicité"].groupby("id_boutique").montant.sum()
                      .to_frame("publicite").join(reel[reel.famille_compte == "Ventes"].groupby("id_boutique").montant.sum().rename("ventes")))
publicite_boutique = publicite_boutique.join(dim_boutique.set_index("id_boutique")[["nom_boutique", "type_boutique"]])
print("Publicité sur trois ans : les trois boutiques qui vendent le plus et les trois qui vendent le moins")
display(pd.concat([publicite_boutique.nlargest(3, "ventes"), publicite_boutique.nsmallest(3, "ventes")])
        [["nom_boutique", "type_boutique", "ventes", "publicite"]].style.format({"ventes": "{:,.0f}", "publicite": "{:,.0f}"}))

noter("Scénarios construits à partir du réel (0 = budget et prévision indépendants du réel)",
      int(rapports["prévision ÷ réel"].round(2).value_counts(normalize=True).iloc[0] > 0.5), "Information",
      "Présenter l'écart réel / budget comme une illustration de méthode, pas comme une vraie performance")

### Les charges suivent-elles les ventes ? Année par année

Sur l'ensemble des trois années, les frais généraux paraissent proportionnels aux ventes (corrélation proche de 1). Cette corrélation est trompeuse : elle est portée par l'écart de taille entre la boutique en ligne, le revendeur et les magasins. On refait donc le test **par type de boutique et par année**, puis on regarde si les frais généraux d'un mois ressemblent davantage aux ventes du mois ou aux frais généraux du même mois de l'année précédente.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Poids des charges dans les ventes, par type de boutique et par année
# ---------------------------------------------------------------------------
LIBELLES_TYPES = {"Online": "En ligne", "Reseller": "Revendeur", "Store": "Magasins"}
suivi_annee = (plan_mois.assign(annee=plan_mois.date.dt.year)
               .merge(dim_boutique[["id_boutique", "type_boutique"]], on="id_boutique", how="left"))
suivi_annee["Publicité"] = suivi_annee["Publicité"].fillna(0)
lignes = []
for (type_boutique, annee), g in suivi_annee.groupby(["type_boutique", "annee"]):
    lignes.append({"type": LIBELLES_TYPES[type_boutique], "annee": annee,
                   "ventes (M$)": g["Ventes"].sum() / 1e6,
                   "frais généraux (M$)": g["Frais généraux"].sum() / 1e6,
                   "coût des ventes ÷ ventes": g["Coût des ventes"].sum() / g["Ventes"].sum(),
                   "impôts ÷ ventes": g["Impôts"].sum() / g["Ventes"].sum(),
                   "frais généraux ÷ ventes": g["Frais généraux"].sum() / g["Ventes"].sum(),
                   "publicité ÷ ventes": g["Publicité"].sum() / g["Ventes"].sum()})
poids_annee = pd.DataFrame(lignes).set_index(["type", "annee"])
display(poids_annee.style.format({"ventes (M$)": "{:,.1f}", "frais généraux (M$)": "{:,.1f}", "coût des ventes ÷ ventes": "{:.1%}",
                                  "impôts ÷ ventes": "{:.1%}", "frais généraux ÷ ventes": "{:.1%}", "publicité ÷ ventes": "{:.1%}"}))

# ---------------------------------------------------------------------------
# 2. Taux de frais généraux magasin par magasin, par année : est-il le même partout ?
# ---------------------------------------------------------------------------
magasins_annee = (suivi_annee[suivi_annee.type_boutique == "Store"].groupby(["annee", "id_boutique"])
                  [["Ventes", "Frais généraux"]].sum())
magasins_annee["taux"] = magasins_annee["Frais généraux"] / magasins_annee["Ventes"]
print("Taux de frais généraux des magasins (frais généraux ÷ ventes), dispersion par année :")
display(magasins_annee.groupby("annee").taux.describe(percentiles=[0.05, 0.5, 0.95])
        .style.format({"count": "{:.0f}", "mean": "{:.1%}", "std": "{:.1%}", "min": "{:.1%}", "5%": "{:.1%}",
                       "50%": "{:.1%}", "95%": "{:.1%}", "max": "{:.1%}"}))

# ---------------------------------------------------------------------------
# 3. Les frais généraux sont-ils reconduits d'une année sur l'autre ?
# ---------------------------------------------------------------------------
# Pour chaque boutique, chaque compte et chaque mois du calendrier, on divise le montant de l'année
# par celui du MÊME MOIS de l'année précédente. Si les frais suivaient les ventes, ce rapport
# bougerait comme les ventes (qui chutent). S'il est presque le même partout, c'est que les frais
# de l'année sont ceux de l'année précédente, majorés d'un coefficient.
# On se limite aux magasins déjà ouverts au début de la première des deux années comparées.
annees_plan = sorted(charges.annee.unique())
fg_compte = (reel[reel.famille_compte == "Frais généraux"]
             .pivot_table(index=["id_boutique", "type_boutique", "nom_compte", "mois"], columns="annee", values="montant", aggfunc="sum"))
ventes_bm = (reel[reel.famille_compte == "Ventes"]
             .pivot_table(index=["id_boutique", "type_boutique", "mois"], columns="annee", values="montant", aggfunc="sum"))
ouverture = dim_boutique.set_index("id_boutique").date_ouverture
reconduction = []
for precedente, annee in zip(annees_plan[:-1], annees_plan[1:]):
    deja_ouverts = ouverture[ouverture < pd.Timestamp(precedente, 1, 2)].index
    f = fg_compte[fg_compte.index.get_level_values("id_boutique").isin(deja_ouverts)
                  & (fg_compte.index.get_level_values("type_boutique") == "Store")]
    v = ventes_bm[ventes_bm.index.get_level_values("id_boutique").isin(deja_ouverts)
                  & (ventes_bm.index.get_level_values("type_boutique") == "Store")]
    rapport_ventes = (v[annee] / v[precedente]).dropna()
    reconduction.append({"comparaison": f"{annee} ÷ {precedente}", "compte": "Ventes (pour mémoire)",
                         "médiane": rapport_ventes.median(), "5 %": rapport_ventes.quantile(0.05),
                         "95 %": rapport_ventes.quantile(0.95), "cellules": len(rapport_ventes)})
    for compte, g in f.groupby("nom_compte"):
        rapport = (g[annee] / g[precedente]).dropna()
        reconduction.append({"comparaison": f"{annee} ÷ {precedente}", "compte": compte, "médiane": rapport.median(),
                             "5 %": rapport.quantile(0.05), "95 %": rapport.quantile(0.95), "cellules": len(rapport)})
reconduction = pd.DataFrame(reconduction).set_index(["comparaison", "compte"])
print("Magasins : montant du mois ÷ montant du même mois de l'année précédente (même magasin, même compte) :")
display(reconduction.style.format({"médiane": "{:.3f}", "5 %": "{:.3f}", "95 %": "{:.3f}", "cellules": "{:,.0f}"}))

# Même question en une corrélation : les frais généraux d'un mois-magasin ressemblent-ils plus aux
# ventes du mois, ou aux frais généraux du même mois un an plus tôt ?
fg_bm = fg_compte.groupby(["id_boutique", "type_boutique", "mois"]).sum(min_count=1)
fg_bm = fg_bm[fg_bm.index.get_level_values("type_boutique") == "Store"]
v_bm = ventes_bm[ventes_bm.index.get_level_values("type_boutique") == "Store"]
lignes = [{"annee": annees_plan[0], "corrélation avec les ventes du mois": fg_bm[annees_plan[0]].corr(v_bm[annees_plan[0]]),
           "corrélation avec les frais généraux du même mois, un an plus tôt": np.nan}]
for precedente, annee in zip(annees_plan[:-1], annees_plan[1:]):
    deja_ouverts = ouverture[ouverture < pd.Timestamp(precedente, 1, 2)].index
    masque = fg_bm.index.get_level_values("id_boutique").isin(deja_ouverts)
    lignes.append({"annee": annee, "corrélation avec les ventes du mois": fg_bm.loc[masque, annee].corr(v_bm.loc[masque, annee]),
                   "corrélation avec les frais généraux du même mois, un an plus tôt": fg_bm.loc[masque, annee].corr(fg_bm.loc[masque, precedente])})
print("Frais généraux d'un magasin, un mois donné :")
display(pd.DataFrame(lignes).set_index("annee").style.format("{:+.3f}", na_rep="(première année)"))

# ---------------------------------------------------------------------------
# 4. Publicité : montant par boutique et par année
# ---------------------------------------------------------------------------
publicite_annee = (reel[reel.famille_compte == "Publicité"].groupby(["annee", "id_boutique"]).montant.sum()
                   .groupby("annee").agg(boutiques="size", total="sum", par_boutique_ouverte_toute_l_annee="max"))
publicite_annee["évolution du montant par boutique"] = publicite_annee.par_boutique_ouverte_toute_l_annee.pct_change()
print("Publicité par année (scénario réel) :")
display(publicite_annee.style.format({"total": "{:,.0f}", "par_boutique_ouverte_toute_l_annee": "{:,.0f}",
                                      "évolution du montant par boutique": "{:+.1%}"}, na_rep=""))

noter("Frais généraux reconduits d'une année sur l'autre, sans lien avec les ventes de l'année (1 = oui)", 1, "Information",
      "Traiter les frais généraux des magasins comme des charges fixes dans les analyses")

**Lecture**

- La **prévision** vaut presque toujours le réel multiplié par le même coefficient (1,01) : elle est calculée à partir du réel, pas établie avant lui.
- Le **budget** suit le réel de très près : environ 0,94 fois le réel en 2019, 0,89 fois en 2020 et 0,96 fois en 2021, avec très peu de dispersion d'une boutique et d'un mois à l'autre. Un vrai budget, fixé en début d'année, s'écarterait beaucoup plus du réel.
- Le **coût des ventes**, les **impôts** et les « autres charges » sont une part fixe des ventes, la même les trois années (43,4 %, 10,8 % et 1,1 %).
- **Les frais généraux ne suivent les ventes qu'en 2019.** Cette année-là, ils valent 16,2 % des ventes dans tous les magasins, comme en ligne et chez le revendeur, presque sans dispersion (corrélation de 0,995 avec les ventes du mois). Ensuite, ils sont **reconduits** : chaque compte d'un magasin vaut, chaque mois, le montant du même mois de l'année précédente multiplié par un coefficient fixe, identique pour tous les magasins et au cent près : administration × 1,15, personnel, informatique et énergie × 1,08, loyers × 1,00 en 2020 puis × 1,03 en 2021. Seules les « autres charges » continuent de suivre les ventes. Pendant ce temps les ventes des mêmes magasins, mois pour mois, reculent de 45 % puis de 15 % (médiane). Les frais généraux d'un mois ressemblent donc à ceux du même mois un an plus tôt (corrélation de 0,97 puis 1,00), plus qu'aux ventes du mois (0,58 puis 0,45). Le taux de frais généraux des magasins passe ainsi de 16,2 % à 28,9 % puis 36,3 % des ventes : ce n'est pas le taux qui a été relevé, c'est le chiffre d'affaires qui a fondu sous des charges fixes.
- Un magasin ouvert pendant la période reçoit, l'année de son ouverture, le taux de 16,2 % sur ses propres ventes, puis la même reconduction. C'est ce qui explique que les magasins récents aient un taux bas en 2020 et 2021.
- La **publicité** ne dépend pas des ventes de la boutique : chaque boutique reçoit le même montant, qu'elle vende 5 millions ou 270 millions par an. Ce montant baisse d'une année à l'autre : 1,54 M$ par boutique en 2019, 1,37 M$ en 2020 (−11 %), 1,17 M$ en 2021 (−15 %). La baisse est bien plus faible que celle des ventes des magasins.
- Le plan est donc lui aussi **généré**, par deux règles simples : une part des ventes pour ce qui est variable, un report de l'année précédente pour ce qui est fixe. Un écart « réel contre budget » se calcule sans difficulté, mais il ne mesure pas la qualité de gestion d'une boutique. À dire dans le rapport si on présente cet écart.
- **Ce que cela apporte à l'analyse** : la structure des charges des magasins est celle d'un commerce réel, avec des charges fixes dimensionnées pour les ventes de 2019. Avec 2019 comme année de référence, la question « pourquoi les magasins perdent-ils de l'argent en 2021 ? » devient mesurable : il faut expliquer pourquoi les ventes ont baissé, puisque les charges, elles, n'ont pas suivi (section 7 et notebooks d'analyse).

## 5. Nettoyage : décisions

| Constat | Décision |
|---|---|
| Aucune valeur manquante, aucun doublon, aucun montant négatif | Rien à corriger |
| Six magasins ouverts pendant la période, avec des charges sur quelques mois d'avant leur ouverture | On garde ces montants (totaux du plan). La table commune reçoit deux indicateurs, `mois_avant_ouverture` et `annee_complete`, pour comparer les magasins sur leurs années d'ouverture complète |
| Comptes de publicité absents certains mois | Normal : une campagne n'a de dépenses que pendant sa période. Un compte absent vaut 0 dans les totaux |
| Ventes du plan = notre chiffre d'affaires, au cent près | Le plan est rapprochable de nos ventes |
| Coût des ventes du plan = `cout_total` de la source (trois logiques mélangées) | On n'utilise pas le coût des ventes du plan pour nos marges : on garde `cout_brut` et `cout_net` |
| Budget et prévision calculés à partir du réel | On les garde, en signalant la limite |
| Frais généraux reconduits d'une année sur l'autre | Aucune correction : c'est une propriété du plan. Dans les analyses, ils sont traités comme des charges fixes, et les résultats sont lus année par année |
| Clés techniques de Contoso (`EntityKey`, `CurrencyKey`, `DateContoso`) | Non reprises dans la table propre : une seule devise, et l'entité se déduit de la boutique |

**Aucune valeur du plan n'est modifiée.**

## 6. Variables : compte de résultat par boutique, mois et scénario

On passe de la table longue (une ligne par compte) à un **compte de résultat** : une ligne par boutique, mois et scénario, une colonne par famille, puis les soldes.

| Variable | Formule |
|---|---|
| `ventes_plan` | somme du compte Ventes |
| `cout_ventes_plan` | somme du compte Coût des ventes |
| `frais_generaux` | administration + informatique + personnel + énergie et communication + loyers + autres (toutes les charges d'exploitation hors publicité) |
| `publicite` | somme de toutes les campagnes |
| `impots` | compte Impôts et taxes |
| `charges_fixes_et_publicite` | frais généraux + publicité |
| `marge_brute_plan` | ventes − coût des ventes |
| `resultat_avant_impot_plan` | marge brute du plan − frais généraux − publicité |
| `resultat_net_plan` | résultat avant impôt − impôts |

Ces soldes sont ceux **du plan**, calculés avec son propre coût des ventes. Ils servent à comparer les trois scénarios entre eux. La rentabilité nette calculée avec **nos** marges est construite à la section 7.

In [ ]:
# Noms de colonnes des familles dans le compte de résultat
NOMS_FAMILLES = {"Ventes": "ventes_plan", "Coût des ventes": "cout_ventes_plan",
                 "Frais généraux": "frais_generaux", "Publicité": "publicite", "Impôts": "impots"}

# Une ligne par boutique, mois et scénario ; une colonne par famille.
# fill_value=0 : une famille sans dépense ce mois-là (pas de campagne de publicité) vaut 0.
resultat = (charges.pivot_table(index=["id_boutique", "nom_boutique", "type_boutique", "pays", "date", "annee", "mois", "annee_mois", "scenario"],
                                columns="famille_compte", values="montant", aggfunc="sum", fill_value=0)
            .rename(columns=NOMS_FAMILLES).reset_index())
resultat.columns.name = None

# Soldes du compte de résultat
resultat["charges_fixes_et_publicite"] = resultat.frais_generaux + resultat.publicite
resultat["marge_brute_plan"] = resultat.ventes_plan - resultat.cout_ventes_plan
resultat["resultat_avant_impot_plan"] = resultat.marge_brute_plan - resultat.charges_fixes_et_publicite
resultat["resultat_net_plan"] = resultat.resultat_avant_impot_plan - resultat.impots

print(f"Compte de résultat : {len(resultat):,} lignes (boutique × mois × scénario)".replace(",", " "))

# Vue d'ensemble : compte de résultat par année et par scénario
synthese = (resultat.groupby(["annee", "scenario"])[["ventes_plan", "cout_ventes_plan", "marge_brute_plan", "frais_generaux",
                                                      "publicite", "resultat_avant_impot_plan", "impots", "resultat_net_plan"]]
            .sum().T)
display(synthese.style.format("{:,.0f}"))

# Détail des charges par compte, scénario réel, pour voir ce qui pèse le plus
detail = (reel[reel.famille_compte.isin(["Frais généraux", "Publicité", "Impôts"])]
          .groupby(["famille_compte", "nom_compte"]).montant.sum().reset_index())
detail["part_des_ventes"] = detail.montant / reel.loc[reel.famille_compte == "Ventes", "montant"].sum()
detail["famille_compte"] = pd.Categorical(detail.famille_compte, FAMILLES, ordered=True)
detail = detail.sort_values(["famille_compte", "montant"], ascending=[True, False])
display(detail.style.format({"montant": "{:,.0f}", "part_des_ventes": "{:.1%}"}))

## 7. Table commune : ventes et charges par boutique et par mois

C'est la table qui servira aux analyses et aux tableaux de bord. On ramène nos ventes au niveau boutique et mois (avec les variables de `05_variables`), puis on y joint les charges réelles du plan.

**La rentabilité nette, calculée avec nos marges :**

| Variable | Formule | Remarque |
|---|---|---|
| `resultat_avant_publicite` | marge nette − frais généraux | |
| `resultat_exploitation` | résultat avant publicité − publicité | Les frais généraux et la publicité sont tous deux des charges d'exploitation |
| `resultat_net` | résultat d'exploitation − impôts | |
| `taux_resultat_exploitation` | résultat d'exploitation ÷ chiffre d'affaires net | À recalculer à partir des sommes dans tout regroupement |
| `ecart_ventes_budget` | chiffre d'affaires brut − ventes du budget | Voir la limite de la section 4 |
| `mois_avant_ouverture` | vrai si le mois précède le mois d'ouverture de la boutique | Ventes éparses de la règle N8 |
| `annee_complete` | vrai si la boutique était ouverte au 1er janvier de l'année | À utiliser pour toute comparaison entre magasins |

**Attention au vocabulaire.** Dans ce projet, `marge_nette` (définie dans `05_variables`) veut dire « nette des retours » : chiffre d'affaires net des retours moins coût net des retours. C'est une marge **avant** toute charge et avant impôt, et non le résultat net au sens comptable. Le résultat net est ici la colonne `resultat_net`.

On utilise **notre** marge nette et non la marge du plan, parce que le coût des ventes du plan mélange trois logiques (section 3). L'écart entre les deux résultats est affiché pour mémoire.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Nos ventes au niveau boutique et mois
# ---------------------------------------------------------------------------
ventes_boutique_mois = (ventes.groupby(["id_boutique", "date_mois"])
                        .agg(nb_lignes=("id_vente", "size"),
                             quantite_vente=("quantite_vente", "sum"),
                             quantite_retour=("quantite_retour", "sum"),
                             montant_avant_remise=("montant_avant_remise", "sum"),
                             montant_remise=("montant_remise", "sum"),
                             montant_retour=("montant_retour", "sum"),
                             ca_brut=("ca_brut", "sum"), ca_net=("ca_net", "sum"),
                             cout_brut=("cout_brut", "sum"), cout_net=("cout_net", "sum"),
                             marge_brute=("marge_brute", "sum"), marge_nette=("marge_nette", "sum"))
                        .reset_index().rename(columns={"date_mois": "date"}))

# ---------------------------------------------------------------------------
# 2. Les charges réelles et les repères du budget, au même niveau
# ---------------------------------------------------------------------------
cles = ["id_boutique", "date"]
charges_reelles = resultat.loc[resultat.scenario == "Réel",
                               cles + ["nom_boutique", "type_boutique", "pays", "annee", "mois", "annee_mois",
                                       "frais_generaux", "publicite", "charges_fixes_et_publicite", "impots",
                                       "cout_ventes_plan", "resultat_avant_impot_plan", "resultat_net_plan"]]
budget = (resultat.loc[resultat.scenario == "Budget", cles + ["ventes_plan", "charges_fixes_et_publicite", "resultat_avant_impot_plan"]]
          .rename(columns={"ventes_plan": "ventes_budget", "charges_fixes_et_publicite": "charges_fixes_et_publicite_budget",
                           "resultat_avant_impot_plan": "resultat_avant_impot_budget"}))

# validate="one_to_one" : exactement une ligne par boutique et par mois de chaque côté
commun = (ventes_boutique_mois
          .merge(charges_reelles, on=cles, how="left", validate="one_to_one")
          .merge(budget, on=cles, how="left", validate="one_to_one"))

# ---------------------------------------------------------------------------
# 3. Rentabilité nette, avec NOS marges
# ---------------------------------------------------------------------------
commun["resultat_avant_publicite"] = commun.marge_nette - commun.frais_generaux
commun["resultat_exploitation"] = commun.resultat_avant_publicite - commun.publicite
commun["resultat_net"] = commun.resultat_exploitation - commun.impots
commun["taux_resultat_exploitation"] = commun.resultat_exploitation / commun.ca_net
commun["taux_remise_effectif"] = commun.montant_remise / commun.montant_avant_remise
commun["taux_retour"] = commun.quantite_retour / commun.quantite_vente
commun["ecart_ventes_budget"] = commun.ca_brut - commun.ventes_budget

# Indicateurs d'activité : mois d'avant l'ouverture, et année d'ouverture complète
date_ouverture = commun.id_boutique.map(dim_boutique.set_index("id_boutique").date_ouverture)
commun["mois_avant_ouverture"] = commun.date < date_ouverture.dt.to_period("M").dt.to_timestamp()
commun["annee_complete"] = date_ouverture < pd.to_datetime(commun.annee.astype(str) + "-01-02")

# Colonnes dans un ordre lisible
commun = commun[["date", "annee", "mois", "annee_mois", "id_boutique", "nom_boutique", "type_boutique", "pays",
                 "nb_lignes", "quantite_vente", "quantite_retour", "montant_avant_remise", "montant_remise", "montant_retour",
                 "ca_brut", "ca_net", "cout_brut", "cout_net", "marge_brute", "marge_nette",
                 "frais_generaux", "publicite", "charges_fixes_et_publicite", "impots",
                 "resultat_avant_publicite", "resultat_exploitation", "resultat_net", "taux_resultat_exploitation", "taux_remise_effectif", "taux_retour",
                 "cout_ventes_plan", "resultat_avant_impot_plan", "resultat_net_plan",
                 "ventes_budget", "charges_fixes_et_publicite_budget", "resultat_avant_impot_budget", "ecart_ventes_budget",
                 "mois_avant_ouverture", "annee_complete"]]
print(f"Table commune : {len(commun):,} lignes (boutique × mois) × {commun.shape[1]} colonnes".replace(",", " "))

# Écart entre notre résultat d'exploitation et le résultat avant impôt du plan : il vient uniquement
# du coût (cout_net chez nous, cout_total dans le plan) et des retours (déduits chez nous).
print(f"Résultat d'exploitation (nos marges)  : {commun.resultat_exploitation.sum():,.0f} $".replace(",", " "))
print(f"Résultat avant impôt du plan           : {commun.resultat_avant_impot_plan.sum():,.0f} $".replace(",", " "))
print(f"Écart                                  : {(commun.resultat_exploitation - commun.resultat_avant_impot_plan).sum():,.0f} $".replace(",", " "))

# ---------------------------------------------------------------------------
# 4. Lecture par boutique et par année
# ---------------------------------------------------------------------------
SOLDES = ["ca_net", "marge_nette", "frais_generaux", "publicite", "resultat_exploitation", "impots", "resultat_net"]
par_boutique_annee = (commun.groupby(["annee", "id_boutique", "nom_boutique", "type_boutique", "pays"])
                      .agg(**{c: (c, "sum") for c in SOLDES}, nb_mois=("date", "nunique"),
                           annee_complete=("annee_complete", "first")).reset_index())
# Taux recalculé à partir des sommes, jamais comme moyenne des taux mensuels
par_boutique_annee["taux_resultat_exploitation"] = par_boutique_annee.resultat_exploitation / par_boutique_annee.ca_net
# Même table sur l'ensemble de la période (utilisée plus bas pour les moyens des boutiques)
par_boutique = (commun.groupby(["id_boutique", "nom_boutique", "type_boutique", "pays"])[SOLDES].sum().reset_index())
par_boutique["taux_resultat_exploitation"] = par_boutique.resultat_exploitation / par_boutique.ca_net

format_boutique = {c: "{:,.0f}" for c in SOLDES}
format_boutique["taux_resultat_exploitation"] = "{:.1%}"
exercice = par_boutique_annee[par_boutique_annee.annee == EXERCICE].sort_values("resultat_exploitation", ascending=False)
colonnes_affichees = ["nom_boutique", "type_boutique", "nb_mois"] + SOLDES + ["taux_resultat_exploitation"]
print(f"Exercice {EXERCICE} : les cinq boutiques au résultat d'exploitation le plus élevé :")
display(exercice.head(5)[colonnes_affichees].style.format(format_boutique).hide(axis="index"))
print(f"Exercice {EXERCICE} : les cinq boutiques au résultat d'exploitation le plus faible :")
display(exercice.tail(5)[colonnes_affichees].style.format(format_boutique).hide(axis="index"))

# Magasins bénéficiaires, année par année. On compte à part les magasins ouverts toute l'année :
# un magasin ouvert en cours d'année porte des charges sur des mois où il ne vend presque rien.
magasins_annee = par_boutique_annee[par_boutique_annee.type_boutique == "Store"]
decompte = magasins_annee.groupby("annee").apply(lambda g: pd.Series({
    "magasins avec des ventes": len(g),
    "ouverts toute l'année": int(g.annee_complete.sum()),
    "dont résultat d'exploitation positif": int((g.annee_complete & (g.resultat_exploitation > 0)).sum()),
    "dont résultat net positif": int((g.annee_complete & (g.resultat_net > 0)).sum()),
    "ouverts en cours d'année ou plus tard": int((~g.annee_complete).sum()),
    "résultat d'exploitation de ces derniers (M$)": g.loc[~g.annee_complete, "resultat_exploitation"].sum() / 1e6,
}), include_groups=False)
print("Magasins bénéficiaires, par année :")
display(decompte.style.format({c: "{:.0f}" for c in decompte.columns[:5]} | {decompte.columns[5]: "{:+.1f}"}))
print("Mois-boutiques au résultat d'exploitation négatif, par année :",
      commun.groupby("annee").apply(lambda g: f"{int((g.resultat_exploitation < 0).sum())} sur {len(g)}", include_groups=False).to_dict())

# ---------------------------------------------------------------------------
# 5. Cascade du résultat par type de boutique et par année
# ---------------------------------------------------------------------------
# De haut en bas : ce qui reste du chiffre d'affaires après chaque étage de charges.
ETAGES = [("ca_net", "Chiffre d'affaires net des retours"),
          ("marge_nette", "Marge nette des retours"),
          ("frais_generaux", "− Frais généraux"),
          ("resultat_avant_publicite", "= Résultat avant publicité"),
          ("publicite", "− Publicité"),
          ("resultat_exploitation", "= Résultat d'exploitation"),
          ("impots", "− Impôts"),
          ("resultat_net", "= Résultat net")]
ORDRE_TYPES = ["Magasins", "En ligne", "Revendeur", "Ensemble"]

par_type_annee = commun.groupby(["type_boutique", "annee"])[[c for c, _ in ETAGES]].sum().rename(index=LIBELLES_TYPES, level=0)
ensemble = commun.groupby("annee")[[c for c, _ in ETAGES]].sum()
ensemble.index = pd.MultiIndex.from_product([["Ensemble"], ensemble.index], names=["type_boutique", "annee"])
par_type_annee = pd.concat([par_type_annee, ensemble]).loc[ORDRE_TYPES]

# a) En millions de dollars
cascade = (par_type_annee / 1e6).T
cascade.index = [libelle for _, libelle in ETAGES]
print("\nCascade du résultat par type de boutique et par année (millions de dollars) :")
display(cascade.style.format("{:,.1f}"))

# b) En pourcentage du chiffre d'affaires net : c'est la lecture la plus parlante,
#    parce qu'elle compare des boutiques de tailles très différentes.
ratios = par_type_annee.div(par_type_annee.ca_net, axis=0).T
ratios.index = [libelle for _, libelle in ETAGES]
print("Même cascade, en pourcentage du chiffre d'affaires net :")
display(ratios.style.format("{:.1%}"))

# c) Les magasins entre l'année de référence et l'exercice : d'où vient la variation du résultat ?
m = par_type_annee.loc["Magasins"]
pont = pd.DataFrame({
    f"{REFERENCE}": m.loc[REFERENCE], f"{EXERCICE}": m.loc[EXERCICE],
    "variation": m.loc[EXERCICE] - m.loc[REFERENCE]}).loc[["ca_net", "marge_nette", "frais_generaux", "publicite", "resultat_exploitation"]] / 1e6
pont["évolution"] = pont["variation"] / pont[f"{REFERENCE}"]
pont.index = ["Chiffre d'affaires net", "Marge nette des retours", "Frais généraux", "Publicité", "Résultat d'exploitation"]
print(f"Magasins, de {REFERENCE} à {EXERCICE} (millions de dollars) :")
display(pont.style.format({f"{REFERENCE}": "{:,.1f}", f"{EXERCICE}": "{:,.1f}", "variation": "{:+,.1f}", "évolution": "{:+.1%}"}))

In [ ]:
# ---------------------------------------------------------------------------
# Graphique : ce que deviennent 100 $ de chiffre d'affaires
# ---------------------------------------------------------------------------
# Une barre par situation. Trois parts empilées : le coût des produits, les frais généraux et la
# publicité, en pourcentage du chiffre d'affaires net. Le trait vertical marque 100 % : ce qui reste
# à gauche du trait est le résultat d'exploitation ; une barre qui le dépasse est en déficit.
situations = ([("Magasins", a) for a in sorted(commun.annee.unique())]
              + [("En ligne", EXERCICE), ("Revendeur", EXERCICE)])
decoupe = pd.DataFrame({
    "Coût des produits vendus": par_type_annee.ca_net - par_type_annee.marge_nette,
    "Frais généraux": par_type_annee.frais_generaux,
    "Publicité": par_type_annee.publicite,
}).div(par_type_annee.ca_net, axis=0).loc[situations]
resultat_part = (par_type_annee.resultat_exploitation / par_type_annee.ca_net).loc[situations]

fig, ax = plt.subplots(figsize=(10.5, 4.4))
etiquettes_lignes = [f"{nom} {annee}" for nom, annee in situations][::-1]
couleurs_parts = [viz.GRIS, viz.CATEGORIES[2], viz.ALERTE]
gauche = np.zeros(len(decoupe))
for (nom_part, valeurs), couleur in zip(decoupe.iloc[::-1].items(), couleurs_parts):
    ax.barh(etiquettes_lignes, valeurs.values, left=gauche, color=couleur, label=nom_part, edgecolor="white", linewidth=1)
    for y, (debut, valeur) in enumerate(zip(gauche, valeurs.values)):
        if valeur >= 0.03:                       # pas d'étiquette sur les parts trop fines pour la contenir
            ax.text(debut + valeur / 2, y, f"{valeur:.1%}".replace(".", ","), ha="center", va="center", color="white", fontweight="bold")
    gauche = gauche + valeurs.values
ax.axvline(1, color=viz.PRINCIPALE, linewidth=1.5, linestyle="--")
for y, (total, reste) in enumerate(zip(gauche, resultat_part.values[::-1])):
    ax.text(max(total, 1) + 0.012, y, f"résultat : {reste:+.1%}".replace(".", ","), va="center",
            color=viz.ALERTE if reste < 0 else viz.PRINCIPALE, fontweight="bold")
ax.set_xlim(0, 1.27)
ax.set_xticks(np.arange(0, 1.01, 0.2))
viz.format_pourcent(ax, axe="x")
ax.grid(False)
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.42, -0.1), frameon=False)
viz.titre(ax, "Les charges des magasins dépassent leur chiffre d'affaires en " + str(EXERCICE),
          "Coûts en pourcentage du chiffre d'affaires net des retours · trait pointillé : 100 %")
print("Figure enregistrée :", viz.enregistrer(fig, "05b_structure_resultat"))
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Graphique : résultat d'exploitation des magasins sur l'exercice, comparé à l'année de référence
# ---------------------------------------------------------------------------
# La boutique en ligne et le revendeur sont exclus du graphique : leurs résultats (une centaine de
# millions chacun) écraseraient l'échelle. Leurs montants sont rappelés dans le sous-titre.
magasins = (par_boutique_annee[(par_boutique_annee.type_boutique == "Store") & (par_boutique_annee.annee == EXERCICE)]
            .sort_values("resultat_exploitation"))
reference_magasins = (par_boutique_annee[(par_boutique_annee.annee == REFERENCE) & par_boutique_annee.annee_complete]
                      .set_index("id_boutique").resultat_exploitation)
hors_magasins = (par_boutique_annee[(par_boutique_annee.type_boutique != "Store") & (par_boutique_annee.annee == EXERCICE)]
                 .set_index("type_boutique").resultat_exploitation)

fig, ax = plt.subplots(figsize=(9, 9.5))
noms = magasins.nom_boutique.str.replace("Contoso ", "", regex=False).str.replace(" Store", "", regex=False)
couleurs = np.where(magasins.resultat_exploitation < 0, viz.ALERTE, viz.PRINCIPALE)
ax.barh(noms, magasins.resultat_exploitation, color=couleurs)
# Repère : le résultat du même magasin l'année de référence (magasins alors ouverts toute l'année)
ax.scatter(magasins.id_boutique.map(reference_magasins), noms, color=viz.GRIS, s=18, zorder=3)
ax.axvline(0, color=viz.GRIS, linewidth=1)
viz.format_dollars(ax, axe="x")
ax.tick_params(axis="y", labelsize=8)
ax.margins(y=0.01)
# Légende construite à la main : les barres ont deux couleurs selon le signe du résultat
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=viz.PRINCIPALE, label=f"Bénéfice {EXERCICE}"), Patch(color=viz.ALERTE, label=f"Déficit {EXERCICE}"),
                   Line2D([], [], marker="o", linestyle="", color=viz.GRIS, markersize=5,
                          label=f"Résultat {REFERENCE} (magasins alors ouverts toute l'année)")],
          loc="center", bbox_to_anchor=(0.42, 0.5), frameon=False)
# Pas de signe dollar dans le sous-titre : matplotlib lirait le texte entre deux signes comme une formule
rappel = ", ".join(f"{'en ligne' if t == 'Online' else 'revendeur'} {v / 1e6:,.0f} M".replace(",", " ")
                   for t, v in hors_magasins.items())
viz.titre(ax, f"Résultat d'exploitation des magasins en {EXERCICE}",
          f"Marge nette moins frais généraux et publicité · points gris : {REFERENCE} · hors graphique : {rappel}")
ax.texts[-1].set_y(1.006)   # la figure est haute : on rapproche le sous-titre de l'axe pour qu'il ne touche pas le titre
print("Figure enregistrée :", viz.enregistrer(fig, "05b_resultat_par_boutique"))
plt.show()

**Lecture**

- **En 2019, tous les magasins gagnent de l'argent.** Les 48 magasins ouverts toute l'année ont un résultat d'exploitation positif, et les magasins dégagent ensemble 121,6 M$, soit 23,8 % de leur chiffre d'affaires. Leur structure de coûts est alors celle de la boutique en ligne et du revendeur (56 % de marge, 16,4 % de frais généraux), à une différence près : la publicité, 16 % du chiffre d'affaires contre moins de 1 %.
- **En 2021, ils en perdent.** Le résultat d'exploitation des magasins est de −10,8 M$ (−4,1 % du chiffre d'affaires) et seuls 4 des 53 magasins ouverts toute l'année sont bénéficiaires. Après impôts, il n'en reste qu'un. 2020 est l'année de bascule : +13,0 M$, 43 magasins bénéficiaires sur 51 avant impôts, 5 après.
- **Ce n'est pas la marge qui a changé** : elle reste à 56 % du chiffre d'affaires les trois années, dans les trois canaux. Les prix, les coûts d'achat et le poids des remises et des retours ne sont donc pas en cause à ce niveau.
- **C'est le chiffre d'affaires qui a fondu sous des charges qui n'ont pas suivi.** Entre 2019 et 2021, les magasins perdent 48,6 % de leur chiffre d'affaires et 139,2 M$ de marge. Dans le même temps, leurs frais généraux augmentent de 12,9 M$ (+15,4 %) et la publicité ne baisse que de 19,8 M$ (−24,2 %). Le résultat recule de 132,4 M$.
- **La boutique en ligne et le revendeur tiennent**, parce que leur chiffre d'affaires tient : +6,7 % en ligne entre 2019 et 2021, −14,1 % pour le revendeur. Leurs frais généraux montent eux aussi (de 16,3 % à 18,7 % du chiffre d'affaires en ligne, de 16,5 % à 21,8 % chez le revendeur), sans entamer sérieusement leur résultat. En 2021, le résultat d'exploitation de l'entreprise (141,7 M$) vient entièrement de ces deux boutiques.
- **Les impôts** du plan sont calculés en proportion des ventes et non du résultat : une boutique paie 10,9 % de son chiffre d'affaires, qu'elle gagne ou qu'elle perde de l'argent. C'est ce qui rend le résultat net des magasins négatif dès 2020.
- **Les six magasins ouverts pendant la période** portent en 2019 des charges de publicité complètes (7,5 M$) pour des ventes presque nulles avant leur ouverture (1,1 M$) : c'est une anomalie du plan, qui explique leur résultat négatif cette année-là (−4,1 M$ à eux six). Elle ne touche pas l'exercice étudié.
- Ce constat n'apparaît pas dans la marge : en marge nette des retours, tous les magasins sont bénéficiaires, les trois années. Il faut les charges pour le voir, et il faut l'année de référence pour voir que la situation de 2021 n'est pas l'état normal des magasins. C'est l'apport de cette deuxième table de faits pour la question Q1.

La section suivante cherche à expliquer les écarts de charges entre les canaux à partir des données.

### Pourquoi les frais généraux et la publicité pèsent-ils si peu en ligne et chez le revendeur ?

Dans un vrai commerce, on penserait à une explication simple : une boutique en ligne n'a ni vitrine ni vendeurs, donc moins de loyers et de personnel. On vérifie si les données soutiennent cette explication, avec quatre tests :

1. **La publicité** : comment est-elle répartie entre les boutiques ?
2. **Les frais généraux** : quel taux par boutique et par année, et y a-t-il des magasins au même taux que la boutique en ligne ?
3. **Le lien avec les moyens réels** : le personnel suit-il l'effectif, les loyers suivent-ils la surface ?
4. **La composition** : la boutique en ligne paie-t-elle moins de loyers et de personnel que les autres, ou moins de tout ?

In [ ]:
# ---------------------------------------------------------------------------
# 1. Publicité : le montant mensuel est-il le même pour toutes les boutiques ?
# ---------------------------------------------------------------------------
publicite_mois = (reel[reel.famille_compte == "Publicité"].groupby(["date", "id_boutique"]).montant.sum().reset_index())
dispersion = publicite_mois.groupby("date").montant.agg(boutiques="size", minimum="min", maximum="max", valeurs_differentes="nunique")
print("Publicité par boutique, mois par mois : nombre de montants différents entre les boutiques")
print("  mois où toutes les boutiques ont exactement le même montant :",
      int((dispersion.valeurs_differentes == 1).sum()), "sur", len(dispersion))
display(dispersion.head(6).style.format({"minimum": "{:,.2f}", "maximum": "{:,.2f}"}))

# ---------------------------------------------------------------------------
# 2. Frais généraux : taux par boutique et par année (frais généraux ÷ chiffre d'affaires net)
# ---------------------------------------------------------------------------
taux_annee = par_boutique_annee[["annee", "id_boutique", "nom_boutique", "type_boutique", "nb_mois", "annee_complete",
                                 "ca_net", "frais_generaux", "publicite"]].copy()
taux_annee["taux_frais_generaux"] = taux_annee.frais_generaux / taux_annee.ca_net
taux_annee["taux_publicite"] = taux_annee.publicite / taux_annee.ca_net
# À partir de 2020, deux groupes de magasins apparaissent : on les sépare à 28 % pour les compter
taux_annee["groupe"] = np.where(taux_annee.taux_frais_generaux < 0.28, "taux bas (moins de 28 %)", "taux haut (28 % et plus)")
print("Taux de frais généraux par boutique : répartition par année")
display(taux_annee.groupby(["annee", "groupe", "type_boutique"]).agg(boutiques=("id_boutique", "size"),
                                                                      taux_minimum=("taux_frais_generaux", "min"),
                                                                      taux_maximum=("taux_frais_generaux", "max"),
                                                                      ca_net_median=("ca_net", "median"))
        .style.format({"taux_minimum": "{:.1%}", "taux_maximum": "{:.1%}", "ca_net_median": "{:,.0f}"}))
print(f"Les boutiques au taux bas en {EXERCICE}, avec leur taux des années précédentes :")
bas = taux_annee[(taux_annee.annee == EXERCICE) & taux_annee.groupe.str.startswith("taux bas")].id_boutique
historique = (taux_annee[taux_annee.id_boutique.isin(bas)]
              .pivot_table(index=["nom_boutique", "type_boutique"], columns="annee", values=["ca_net", "taux_frais_generaux"]))
display(historique.sort_values(("ca_net", EXERCICE), ascending=False)
        .style.format({c: ("{:,.0f}" if c[0] == "ca_net" else "{:.1%}") for c in historique.columns}, na_rep="(fermé)"))
print("Date d'ouverture de ces boutiques :",
      dim_boutique[dim_boutique.id_boutique.isin(bas)].set_index("nom_boutique").date_ouverture.dt.strftime("%Y-%m-%d").to_dict())

# Taux sur l'ensemble de la période, par boutique : sert au calcul des coûts par employé et par mètre carré (point 3)
taux = (commun.groupby(["id_boutique", "nom_boutique", "type_boutique"])
        .agg(ca_net=("ca_net", "sum"), frais_generaux=("frais_generaux", "sum"), publicite=("publicite", "sum"),
             nb_mois=("date", "nunique")).reset_index())

# ---------------------------------------------------------------------------
# 3. Les charges suivent-elles les moyens réels de la boutique (effectif, surface) ?
# ---------------------------------------------------------------------------
par_compte = (reel[reel.nom_compte.isin(["Personnel", "Loyers et immobilier"])]
              .pivot_table(index="id_boutique", columns="nom_compte", values="montant", aggfunc="sum"))
moyens = par_compte.join(dim_boutique.set_index("id_boutique")[["nom_boutique", "type_boutique", "effectif", "surface"]])
moyens = moyens.join(taux.set_index("id_boutique")[["ca_net", "nb_mois"]])
# Coût de personnel par employé et par an ; loyer par mètre carré et par an
moyens["personnel_par_employe_par_an"] = moyens["Personnel"] / moyens.effectif / (moyens.nb_mois / 12)
moyens["loyer_par_m2_par_an"] = moyens["Loyers et immobilier"] / moyens.surface / (moyens.nb_mois / 12)

magasins_moyens = moyens[moyens.type_boutique == "Store"]
print("Magasins : corrélation entre les charges et les moyens de la boutique")
print(f"  personnel et effectif : {magasins_moyens['Personnel'].corr(magasins_moyens.effectif):+.2f}",
      f"· loyers et surface : {magasins_moyens['Loyers et immobilier'].corr(magasins_moyens.surface):+.2f}",
      f"· personnel et ventes : {magasins_moyens['Personnel'].corr(magasins_moyens.ca_net):+.2f}")
print("Boutique en ligne, revendeur, et médiane des magasins :")
apercu = pd.concat([moyens[moyens.type_boutique != "Store"][["nom_boutique", "effectif", "surface", "personnel_par_employe_par_an", "loyer_par_m2_par_an"]],
                    pd.DataFrame([{"nom_boutique": "Magasins (médiane)", **magasins_moyens[["effectif", "surface", "personnel_par_employe_par_an", "loyer_par_m2_par_an"]].median().to_dict()}])])
display(apercu.style.format({"effectif": "{:,.0f}", "surface": "{:,.0f}", "personnel_par_employe_par_an": "{:,.0f}", "loyer_par_m2_par_an": "{:,.0f}"}))

# ---------------------------------------------------------------------------
# 4. Composition : part de chaque compte dans les ventes, par type de boutique, année de référence et exercice
# ---------------------------------------------------------------------------
for annee in (REFERENCE, EXERCICE):
    reel_annee = reel[reel.annee == annee]
    ventes_type = reel_annee[reel_annee.famille_compte == "Ventes"].groupby("type_boutique").montant.sum()
    composition = (reel_annee[reel_annee.famille_compte == "Frais généraux"]
                   .pivot_table(index="nom_compte", columns="type_boutique", values="montant", aggfunc="sum") / ventes_type)
    composition = composition.rename(columns=LIBELLES_TYPES)
    composition["magasins ÷ en ligne"] = composition["Magasins"] / composition["En ligne"]
    print(f"Frais généraux en part des ventes, compte par compte, {annee} :")
    display(composition.style.format({"En ligne": "{:.1%}", "Revendeur": "{:.1%}", "Magasins": "{:.1%}", "magasins ÷ en ligne": "{:.2f}"}))

**Ce que montrent les données**

- **Publicité : un budget identique pour chaque boutique.** Chaque mois de campagne, toutes les boutiques reçoivent exactement le même montant, au cent près, quelle que soit leur taille. La publicité pèse donc peu en ligne et chez le revendeur uniquement parce qu'elle est divisée par un chiffre d'affaires trente à cinquante-cinq fois plus grand que celui d'un magasin. Ce n'est pas une publicité plus efficace, c'est une répartition à parts égales.
- **Frais généraux : un seul taux en 2019, deux groupes ensuite, et ce n'est pas une question de canal.** En 2019, toutes les boutiques sont entre 14,6 % et 16,8 % de leur chiffre d'affaires, magasins compris. En 2021, la plupart des magasins sont entre 28 % et 42 %, et quelques boutiques restent sous 28 % : la boutique en ligne, le revendeur, mais **aussi des magasins**. Ces magasins sont ceux dont le chiffre d'affaires n'a pas baissé depuis que leurs frais ont été fixés : les magasins ouverts en 2020 et 2021 (Marseille, Berlin, Venezia), et Moscow No.2, dont les ventes ont progressé. Avec Cambridge, ce sont les seuls magasins bénéficiaires de 2021. Vendre en ligne n'est donc pas ce qui explique le taux bas : c'est d'avoir gardé son chiffre d'affaires.
- **Les charges ne suivent pas les moyens des boutiques.** Dans les magasins, le coût du personnel n'est presque pas lié à l'effectif (corrélation de +0,23), ni les loyers à la surface (+0,14). Pour la boutique en ligne, le coût de personnel par employé (1,8 M$ par an) et le loyer au mètre carré atteignent des niveaux impossibles : elle déclare 7 employés.
- **La boutique en ligne ne paie pas moins de loyers ou de personnel que le reste.** En 2019, chaque compte pèse le même pourcentage des ventes dans les trois canaux. En 2021, les magasins paient environ deux fois plus sur chaque compte, y compris l'informatique, qu'on attendrait au contraire plus élevée pour une vente en ligne. L'écart ne vient pas de la nature du canal : il est apparu en deux ans, avec la baisse du chiffre d'affaires des magasins.

**Conclusion.** Les données ne contiennent pas d'explication économique propre au canal. L'explication « pas de vitrine, pas de vendeurs » serait plausible dans la réalité, mais elle n'est pas soutenue par ce jeu de données, et il ne faut pas l'écrire comme un résultat. Ce que les données montrent est plus simple et plus utile : **les frais généraux sont des charges fixes, calées sur les ventes de la première année puis reconduites**. Une boutique qui garde son chiffre d'affaires garde son taux, une boutique qui en perd la moitié voit son taux doubler.

**Ce qu'on peut dire dans le rapport**

- Le poids de la publicité dans le chiffre d'affaires dépend de la taille de la boutique, parce que le budget est le même pour toutes. Une recommandation défendable en découle : répartir la publicité selon le chiffre d'affaires ou selon son effet mesuré, plutôt qu'à parts égales (question Q5, et US36).
- Le déficit des magasins en 2021 est d'abord un problème de **chiffre d'affaires** face à des charges fixes. Toute piste d'action se juge donc à sa capacité à relever les ventes des magasins ou à ramener les charges fixes à leur niveau, et la première question à traiter est : pourquoi les ventes des magasins ont-elles chuté de moitié quand celles de la boutique en ligne ont tenu ? C'est l'objet des notebooks d'analyse.

### Peut-on déduire l'effectif d'un magasin de sa masse salariale ?

Le nettoyage (`04_nettoyage`, règle N7) a dû remplacer l'effectif manquant d'un magasin. L'idée la plus naturelle est de le déduire de ses charges de personnel : masse salariale ÷ coût moyen d'un employé. On vérifie ici si cette déduction tient, sur les magasins dont l'effectif est **connu** (le magasin imputé est écarté du test).

Deux méthodes sont comparées, chacune appliquée à un magasin à partir des autres :

- **Méthode A, par la masse salariale** : charges de personnel du magasin ÷ coût médian d'un employé dans les autres magasins.
- **Méthode B, par la médiane** : effectif médian des autres magasins.

L'erreur est l'écart entre l'effectif estimé et l'effectif réel, en pourcentage de l'effectif réel.

In [ ]:
# Magasins dont l'effectif est connu (on écarte celui que la règle N7 a imputé)
imputes = set(dim_boutique.loc[dim_boutique.effectif_impute, "id_boutique"])
connus = magasins_moyens[~magasins_moyens.index.isin(imputes)].copy()
connus["personnel_par_an"] = connus["Personnel"] / (connus.nb_mois / 12)        # ramené à une année pleine

print(f"Magasins à l'effectif connu : {len(connus)}")
print(f"Corrélation entre charges de personnel par an et effectif : Pearson {connus.personnel_par_an.corr(connus.effectif):+.2f}"
      f" · Spearman {connus.personnel_par_an.corr(connus.effectif, method='spearman'):+.2f}")
cout = connus.personnel_par_employe_par_an
print(f"Coût de personnel par employé et par an : minimum {cout.min():,.0f} $ · médiane {cout.median():,.0f} $ · maximum {cout.max():,.0f} $"
      .replace(",", " "), f"· rapport maximum ÷ minimum : {cout.max() / cout.min():.0f}")

# Chaque magasin est estimé à partir des AUTRES magasins (sans utiliser sa propre valeur)
erreurs = []
for id_boutique, magasin in connus.iterrows():
    autres = connus.drop(index=id_boutique)
    estimation_a = magasin.personnel_par_an / autres.personnel_par_employe_par_an.median()
    estimation_b = autres.effectif.median()
    erreurs.append({"A. par la masse salariale": abs(estimation_a - magasin.effectif) / magasin.effectif,
                    "B. par la médiane": abs(estimation_b - magasin.effectif) / magasin.effectif})
erreurs = pd.DataFrame(erreurs)
comparaison_methodes = pd.DataFrame({"erreur moyenne": erreurs.mean(), "erreur médiane": erreurs.median(),
                                     "magasins estimés à moins de 25 % près": (erreurs < 0.25).mean()})
display(comparaison_methodes.style.format("{:.0%}"))

# Exemple : parmi les magasins dont la masse salariale diffère de moins de 2 %, les deux aux effectifs les plus éloignés
trie = connus.sort_values("personnel_par_an")
proches = trie.personnel_par_an.pct_change().abs() < 0.02
ecart_effectif = trie.effectif.diff().abs().where(proches)
position = int(np.nanargmax(ecart_effectif.values))
paire = trie.iloc[[position - 1, position]]
print("Deux magasins à la masse salariale presque égale :")
display(paire[["nom_boutique", "effectif", "personnel_par_an", "personnel_par_employe_par_an"]]
        .style.format({"effectif": "{:.0f}", "personnel_par_an": "{:,.0f}", "personnel_par_employe_par_an": "{:,.0f}"}).hide(axis="index"))

# Ce que chaque méthode donnerait pour le magasin imputé
for id_boutique in imputes:
    m = magasins_moyens.loc[id_boutique]
    par_masse_salariale = m["Personnel"] / (m.nb_mois / 12) / connus.personnel_par_employe_par_an.median()
    print(f"{m.nom_boutique} : {par_masse_salariale:.0f} employés par la masse salariale · {m.effectif:.0f} retenus par la règle N7")

**Lecture**

- **La masse salariale ne dit presque rien de l'effectif.** La corrélation de rang entre les deux est nulle (Spearman −0,08). La corrélation de Pearson (+0,23) ne tient qu'à un magasin, le plus gros. Le coût de personnel par employé varie de un à dix entre magasins (de 5 100 $ à 50 000 $ par an).
- **Aucune des deux méthodes n'est fiable, et la déduction par la masse salariale ne fait pas mieux que la médiane.** Elle se trompe de plus de 25 % pour près de trois magasins sur quatre, et son erreur moyenne est plus élevée (55 % contre 46 %). Son erreur médiane est un peu plus faible (38 % contre 40 %) : on ne peut donc pas dire qu'une méthode domine l'autre.
- **Pour le magasin concerné, le choix change peu** : les deux méthodes donnent presque la même valeur. La règle N7 retient la médiane des magasins comparables parce qu'elle est la plus simple à justifier et qu'elle ne s'appuie pas sur un lien qui n'existe pas.
- **Conséquence pour la suite du projet** : toute analyse de l'effectif (productivité par employé, optimisation du nombre d'employés) repose sur une variable qui n'est cohérente ni avec la masse salariale, ni avec les ventes, ni avec la surface. Pour raisonner sur le personnel, il faut utiliser les **charges de personnel**, qui suivent les ventes, et non le nombre d'employés.

## 8. Validation

On vérifie que les tables produites sont cohérentes entre elles et avec la source. Si un contrôle échoue, rien n'est écrit.

In [ ]:
controles = {
    "Table des charges : même nombre de lignes que le fichier extrait": len(charges) == len(plan_brut),
    "Table des charges : même total que le fichier extrait": abs(charges.montant.sum() - plan_brut.Montant.sum()) < TOLERANCE,
    "Table des charges : aucun vide": bool(charges.notna().all().all()),
    "Dimension catégorie : une ligne par catégorie, toutes celles des charges et des produits": bool(
        dim_categorie.id_categorie.is_unique and dim_categorie.categorie.is_unique
        and set(charges.id_categorie) <= set(dim_categorie.id_categorie)
        and categories_produits <= set(dim_categorie.categorie)),
    "Compte de résultat : la somme des familles redonne le total des charges": abs(
        resultat[list(NOMS_FAMILLES.values())].sum().sum() - charges.montant.sum()) < 1,
    "Compte de résultat : résultat net = ventes − coût − exploitation − publicité − impôts": bool((
        (resultat.resultat_net_plan - (resultat.ventes_plan - resultat.cout_ventes_plan - resultat.frais_generaux
                                       - resultat.publicite - resultat.impots)).abs() < TOLERANCE).all()),
    "Table commune : une ligne par boutique et par mois de vente": len(commun) == ventes.groupby(["id_boutique", "date_mois"]).ngroups,
    "Table commune : chiffre d'affaires brut égal à celui de la table des ventes": abs(commun.ca_brut.sum() - ventes.ca_brut.sum()) < 1,
    "Table commune : marge nette égale à celle de la table des ventes": abs(commun.marge_nette.sum() - ventes.marge_nette.sum()) < 1,
    "Table commune : chaque mois-boutique a ses charges (aucun vide)": bool(commun.notna().all().all()),
    "Table commune : charges égales à celles du plan (scénario réel)": abs(
        commun.charges_fixes_et_publicite.sum()
        - reel.loc[reel.famille_compte.isin(["Frais généraux", "Publicité"]), "montant"].sum()) < 1,
    "Rapprochement : ventes du plan égales à notre chiffre d'affaires, par boutique et par mois": bool((ecart_ventes <= TOLERANCE).all()),
}
for nom, ok in controles.items():
    print(f"[{'réussi' if ok else 'ÉCHOUÉ'}] {nom}")
print(f"\n{sum(controles.values())} contrôles réussis sur {len(controles)}")
assert all(controles.values()), "Validation échouée : rien n'est écrit."

print("\nRegistre du diagnostic :")
display(pd.DataFrame(registre))

## 9. Enregistrement

In [ ]:
config.CLEAN.mkdir(parents=True, exist_ok=True)
config.CLEAN_DIMENSIONS.mkdir(parents=True, exist_ok=True)
config.AGREGATS.mkdir(parents=True, exist_ok=True)

sorties = {
    # Deuxième table de faits : une ligne par boutique, mois, scénario, compte et catégorie
    config.CLEAN / "fait_charges.parquet": charges,
    # Ses deux dimensions propres
    config.CLEAN_DIMENSIONS / "dim_compte.parquet": dim_compte,
    config.CLEAN_DIMENSIONS / "dim_scenario.parquet": dim_scenario,
    # Dimension partagée avec les ventes : une ligne par catégorie de produit
    config.CLEAN_DIMENSIONS / "dim_categorie.parquet": dim_categorie,
    # Compte de résultat du plan : une ligne par boutique, mois et scénario
    config.AGREGATS / "resultat_boutique_mois.parquet": resultat,
    # Table commune ventes et charges : une ligne par boutique et par mois
    config.AGREGATS / "ventes_charges_boutique_mois.parquet": commun,
}
for chemin, table in sorties.items():
    table.to_parquet(chemin, index=False)
    print(f"{chemin.relative_to(config.ROOT)}  ({len(table):,} lignes × {table.shape[1]} colonnes)".replace(",", " "))

**Suite** : les analyses lisent `data/clean/agregats/ventes_charges_boutique_mois.parquet` pour la rentabilité nette des boutiques (US35) et `data/clean/fait_charges.parquet` pour le détail de la publicité (US36).

### Le modèle : ventes et charges

```
                        dim_promotion        dim_canal
                              │                  │
                              └────────┐ ┌───────┘
                                       │ │
   dim_produit ─────────────────── fait_ventes ──────────────┐
        │                              │                     │
        │ (colonne categorie)          │                     │
        ▼                              ▼                     ▼
   dim_categorie                  dim_boutique            dim_date
        ▲                              ▲                     ▲
        │                              │                     │
        └───────────────────────── fait_charges ─────────────┘
                                       │ │
                              ┌────────┘ └───────┐
                              │                  │
                         dim_compte         dim_scenario
```

| Table de faits | Se relie à | Par la colonne |
|---|---|---|
| `fait_ventes` | `dim_boutique` | `id_boutique` |
| | `dim_date` | `date` (le jour de la vente) |
| | `dim_produit` | `id_produit` |
| | `dim_promotion` | `id_promotion` |
| | `dim_canal` | `id_canal` |
| `fait_charges` | `dim_boutique` | `id_boutique` |
| | `dim_date` | `date` (le premier jour du mois) |
| | `dim_categorie` | `id_categorie` |
| | `dim_compte` | `id_compte` |
| | `dim_scenario` | `id_scenario` |
| `dim_produit` | `dim_categorie` | `categorie` (le nom de la catégorie) |

Trois dimensions sont **communes** aux deux tables de faits : la boutique, la date et la catégorie de produit. C'est par elles, et seulement par elles, qu'on croise les ventes et les charges.

**Dans Power BI**

- `fait_ventes` et `fait_charges` ne se relient **jamais directement** l'une à l'autre : toutes les relations passent par les dimensions.
- Pour filtrer les deux tables par catégorie, le segment doit venir de `dim_categorie`, pas de `dim_produit`. Un filtre posé sur un produit ou une marque n'a pas de sens pour les charges, qui n'existent qu'au niveau de la catégorie.
- Les charges sont mensuelles : un filtre sur un jour précis ne garde que les charges datées du premier du mois. On analyse donc les charges au mois, au trimestre ou à l'année.
- Les taux (résultat d'exploitation ÷ chiffre d'affaires) sont à définir comme des **mesures**, pas comme des colonnes calculées.

Les trois autres tables de faits (stock, objectifs de vente, ventes en ligne détaillées) sont diagnostiquées dans `05c_stock_objectifs` ; le modèle complet à cinq tables de faits est décrit dans `02_flocon`, section 10.

**Limites à rappeler** : le plan est généré comme les ventes ; le budget et la prévision sont calculés à partir du réel ; les frais généraux sont reconduits d'une année sur l'autre ; le coût des ventes du plan reprend la colonne `cout_total` de la source, que nous n'utilisons pas pour nos marges.